# GeoNUSAF Baseline — SegFormer-B0 **legacy-parity** rebuild

**Structure of `segformer-b0-final.ipynb`. Training logic and hyper-parameters of the earlier
run that reported ~0.63 mIoU** (`Pranilllllll/segformer-satellite-segementation`).

Every optimisation setting below is a literal transcription of the pasted script: 7 classes,
5-fold `KFold(shuffle=True, random_state=42)` over **image indices**, inverse-frequency class
weights over the **whole** dataset, plain weighted cross-entropy, `AdamW(lr=3e-5)` flat with no
scheduler / no warmup / no clipping / no EMA, 300 epochs, patience 25, batch 16, AMP, and the
`HorizontalFlip + VerticalFlip + RandomRotate90 + Resize` albumentations pipeline applied to
**train and val alike**. Section 25 lists the four places where an exact transcription was
impossible and what was done instead.

## Parity table — pasted script vs this notebook
| Setting | Pasted script | Here | Same? |
|---|---|---|---|
| classes | `num_classes = 7`, no ignore index | **6 + `ignore_index=255`** by default | **no — deliberate, see below** |
| split | `KFold(5, shuffle=True, random_state=42)` over image indices | identical | yes |
| class weights | `1/(count+1e-6)`, renormalised, **whole dataset**, uncapped | identical | yes |
| loss | `CrossEntropyLoss(weight=w)` | identical, no Lovasz, no smoothing | yes |
| optimizer | `AdamW(model.parameters(), lr=3e-5)` | identical (single group, torch default `wd=0.01`) | yes |
| scheduler | none | none | yes |
| grad clipping | none | none (norm still logged) | yes |
| EMA | none | none | yes |
| epochs / patience | 300 / 25 | 300 / 25 | yes |
| batch / workers / drop_last | 16 / 2 / False | 16 / 2 / False | yes |
| augmentation | hflip .5, vflip .5, rot90 .5, resize 512 | identical (`albumentations`) | yes |
| val transform | **the train transform** | **clean (resize only)** by default | **no — deliberate** |
| normalisation | `SegformerImageProcessor` | same processor (or verified-equivalent manual path) | yes |
| model regularisation | checkpoint defaults (nothing passed) | nothing passed | yes |
| selection metric | per-batch mIoU, averaged over batches | **`val_miou_core6`** by default | **no — deliberate** |
| image downsample 2345 -> 512 | `A.Resize` bilinear at load time | `INTER_AREA`, cached once | **no — see §25** |

### The three defaults that are no longer parity
All three are scoring choices, not recipe choices — the optimiser, schedule, loss, epochs and
augmentation remain literal. They are set this way because the job has moved on from *reproducing*
the old run to *comparing against the benchmark*, and a comparison needs both sides scored
identically. Restore exact parity with:

```
IGNORE_UNLABELLED = False        # unlabelled becomes class 6 again
VAL_USES_TRAIN_TF = True         # random flips at scoring time
SELECTION_METRIC  = "val_miou_legacy"
```

**Why ignore unlabelled.** Black pixels are annotation *mistakes*, not a land-use category, so
there is no consistent visual signature to learn — and the fold-1 curves bear that out: pooled
7-class ~0.635 against pooled 6-class ~0.72 implies the unlabelled class scored roughly 0.10. It
was the worst class in the run *and* it drew the largest class weight of all seven (3.63 under
inverse frequency, because it is the rarest), so the model spent more gradient per pixel on
annotation noise than on River. Excluding it should move the six real classes slightly upward
rather than merely renaming the metric.

## Read this before you use the 0.63
You asked whether the present code has a bug. Part of that is right, and one part is a real
defect I can name precisely. But most of the 0.63 vs 0.46 gap is **not** in the training code —
it is in how the two runs were split and scored. Ranked by expected size:

**1. Variant leakage (largest).** `KFold` in the pasted script splits the **1224 image files**,
not the 408 base tiles. Each tile exists three times on disk: `imageN.png`, `imageNhf.png`,
`imageNvf.png` — exact mirrors. With 5 folds, a held-out image keeps each of its two siblings in
training with probability 4/5, so **~96% of validation images have an exact mirror of themselves
in the training set**. The model is being scored on scenes it memorised. Section 8 measures this
for your chosen fold and prints the number.

**2. Split geography.** The pasted split is random; the reported protocol is sequence-block. Your
own table already prices this: B0 random 0.4765 vs block 0.3962, a +0.080 gap before any leakage
is counted.

**3. The 7th class.** `num_classes = 7` means the macro mean includes `unlabelled`, which is only
0.365% of pixels but sits in 333 of 408 tiles as large contiguous blobs — geometrically much
easier than an 11 m road. It enters the mean with the same weight as River. How much this is
worth depends entirely on how well that class is learned: if it scores 0.55 against a 0.46 mean it
adds ~0.015, if it scores 0.80 it adds ~0.05. `val_miou_core6` below excludes it so you can read
the answer off rather than estimate it.

**4. Metric definition (smaller than it looks).** `mean_iou()` forms IoU **inside each batch** and
averages over batches, dropping any class whose batch union is 0. That is a mean of ratios where
the confusion matrix gives a ratio of means — a different estimator, not a rescaling of the same
one. I simulated it at your class shares and presence rates: the difference comes out within about
±0.01 and **its sign is not guaranteed** (small-support batches can drag the batch average *down*
as easily as up), and union-0 skipping almost never fires, because a batch of 16 tiles contains
River with probability ~97%. So this is a real defect in the estimator but probably a minor term
in your gap. Both numbers are logged every epoch, so you will not have to take my estimate for it.

**5. Training length.** Your `val_miou` curve is still climbing at the right-hand edge. The
x-axis is the W&B step counter, not the epoch: `log_images_wandb` calls `wandb.log` a second time
every 5 epochs, so 300 epochs renders as ~360 steps. Fold-2 therefore ran the **full 300 epochs
without early stopping and was still improving**. A run that stops at epoch 90 is not the same
experiment.

### The one real bug in the present notebook
`segformer-b0-final.ipynb` builds its train loader with `persistent_workers=True`, and
`train_one_epoch` sets `train_ds.epoch = epoch` in the **parent** process. Persistent workers hold
their own copy of the dataset and never see that assignment, so the augmentation seed
`(SEED, epoch, i)` stays pinned at `epoch=0` for the entire run: every image gets the same rot90
angle and the same brightness jitter in epoch 1 and in epoch 200. **The present B0 run trained on
816 frozen images with no live augmentation.** The pasted script passes no `worker_init_fn` and no
`persistent_workers`, so its augmentation was genuinely fresh every epoch. That is a real
contributor to the gap and it is fixed here (§10).

### About the class weights — the two runs are using almost the same ones
This is worth stating precisely, because it is the part of your hypothesis the arithmetic does not
support. I computed both schemes on your exhaustive pixel counts. Over the 6 real classes:

| | Residential | Road | River | Forest | UnusedLand | Agricultural | max/min |
|---|---|---|---|---|---|---|---|
| legacy `1/n` | 0.042 | 1.499 | 3.921 | 0.227 | 0.181 | 0.130 | 93.4 |
| effective-number, cap 10 | 0.054 | 1.492 | 3.886 | 0.236 | 0.191 | 0.140 | 71.6 |

They agree to within 5% on every class except Residential, where the legacy scheme is 23% more
aggressive. The reason is structural: effective number is `w ∝ 1/(1 - beta^n)`, and once `n` is a
pixel count, `1 - beta^n ≈ n/N` for every class except the dominant one — so it *collapses into*
inverse frequency. The `WEIGHT_CAP = 10` in the reference notebook never even binds; the spread
tops out at 71.6 on its own.

So reweighting is not a difference between the two runs and cannot explain the gap in either
direction. That does not make your observation wrong — heavy rare-class weighting genuinely does
produce speckle, and it would have produced it in the 0.63 run too. It just means the way to test
it is a controlled run, not an inference from the gap. Set `CLASS_WEIGHT_MODE = "none"` in §2,
run one fold, and put the two qualitative figures side by side. If mIoU holds and the speckle goes,
that is an ablation row for the paper rather than a suspicion.

### What to actually run
Three runs, this notebook, one line changed between them (§25 has the recipe). They separate
*protocol* from *recipe* and answer the question the paper needs answered — whether flat 3e-5 for
300 epochs with plain CE genuinely beats the current recipe once both are scored the same way.

## Prerequisites
- **Internet ON**. Kaggle Secrets: `HF_TOKEN` (write) + `WANDB_API_KEY`. GPU: T4 x1.
- One fold per session. Set `FOLD` in §2, run top to bottom.

## 1. Environment

`albumentations` is what the pasted script augments with, so it is imported here rather than
re-implemented in numpy — an approximation of `A.RandomRotate90` is exactly the kind of silent
divergence this notebook exists to avoid. Both it and `transformers` are preinstalled on Kaggle;
only `ptflops` is not.

In [ ]:
!pip install -q ptflops==0.7.3 2>&1 | tail -2

import os, re, gc, json, time, math, random, warnings, itertools, copy
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import albumentations as A
from albumentations.pytorch import ToTensorV2
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| albumentations", A.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Set `FOLD` here, then run the whole notebook.** One fold per session.

`SPLIT_MODE`:
- `"legacy_kfold"` — `KFold(5, shuffle=True, random_state=42)` over the **1224 image indices**.
  This is what the pasted script does, and it lets flip variants of one tile straddle the split.
- `"random"` — random split over the **408 base tiles**, variants grouped. Removes duplicate
  leakage, keeps the random geography.
- `"block"` — contiguous export-order blocks over base tiles. The reported protocol.

Running the same recipe under all three is the decomposition experiment; set `N_FOLDS = 3` for
the latter two so the fold count matches the rest of the benchmark.

`RUN_NAME` carries a `c6` / `c7` tag derived from `IGNORE_UNLABELLED`, because the decode head has
one output channel per class and a 6-way head cannot load a 7-way `last.pt`. Flipping the scheme
therefore starts a clean repo rather than crashing on resume. If you already have runs from before
this tag existed, their repos have no `c6`/`c7` in the name — §21 looks for both spellings.

The three fields under "levers" are not part of the legacy transcription — they are there so a
single-variable test is a one-word edit rather than a code change. Leave them at their defaults
for the parity run.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (change these per session) ----
    SPLIT_MODE: str = "legacy_kfold"   # "legacy_kfold" | "random" | "block"
    FOLD: int = 0                      # 0 .. N_FOLDS-1

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CACHE_DIR: str = "/kaggle/working/cache512"
    PREBUILT_CACHE: Optional[str] = None   # point at the saved cache dataset to skip the repack
    IMG_SIZE: int = 512
    IGNORE_UNLABELLED: bool = True     # True = 6 classes, black pixels excluded from loss+metrics
                                       # False = the legacy 7-class scheme (unlabelled IS a class)
    IGNORE_INDEX: int = 255
    N_CLASSES: int = 6                 # DERIVED below from IGNORE_UNLABELLED -- do not set by hand
    CLASS_NAMES: tuple = ()            # DERIVED below
    PRELOAD_RAM: bool = True

    # ---- split (legacy) ----
    N_FOLDS: int = 5                   # legacy KFold(n_splits=5); use 3 for block/random
    KFOLD_SEED: int = 42               # legacy random_state=42
    BLOCK_BUFFER: int = 2              # only used when SPLIT_MODE == "block"

    # ---- model ----
    HF_MODEL_ID: str = "nvidia/segformer-b0-finetuned-ade-512-512"
    # The legacy script passed NO dropout/drop-path overrides, so the checkpoint's own config
    # values are used verbatim. Do not add them here.

    # ---- optimization: every value below is literal from the pasted script ----
    BATCH_SIZE: int = 16
    LR: float = 3e-5                   # flat, both encoder and head
    WEIGHT_DECAY: float = 0.01         # torch AdamW default; legacy passed none
    MAX_EPOCHS: int = 300
    PATIENCE: int = 25
    AMP: bool = True
    DROP_LAST: bool = False            # legacy DataLoader left this at its default

    # ---- deliberately absent in the legacy script; kept as named constants, not as options ----
    SCHEDULER: Optional[str] = None    # no warmup, no cosine: LR is constant at 3e-5
    CLIP_NORM: Optional[float] = None  # no clip_grad_norm_
    LAMBDA_LOVASZ: float = 0.0         # no Lovasz term
    LABEL_SMOOTHING: float = 0.0
    USE_EMA: bool = False

    # ---- class weights ----
    CLASS_WEIGHT_MODE: str = "legacy_inv_freq"   # "legacy_inv_freq" | "effective_number" | "none"
    WEIGHT_SOURCE: str = "all"                   # legacy counted over train+val together
    WEIGHT_CAP: Optional[float] = None           # legacy applied no cap
    EN_BETA_CAP: float = 10.0                    # only used by "effective_number"

    # ---- augmentation (legacy albumentations pipeline) ----
    AUG_HFLIP_P: float = 0.5
    AUG_VFLIP_P: float = 0.5
    AUG_ROT90_P: float = 0.5
    VAL_USES_TRAIN_TF: bool = False    # legacy was True; see the note in this section
    PREPROCESS: str = "processor"      # "processor" (legacy) | "manual" (verified equivalent)

    # ---- which series selects best.pt ----
    SELECTION_METRIC: str = "val_miou_core6"   # "val_miou_core6" | "val_miou_global" | "val_miou_legacy"

    # ---- levers for single-variable tests; not part of the parity run ----
    # CLASS_WEIGHT_MODE = "none"   -> tests the reweighting hypothesis
    # SPLIT_MODE        = "block"  -> tests the protocol
    # MAX_EPOCHS        = 120      -> tests training length

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 2               # legacy value
    PERSISTENT_WORKERS: bool = False   # legacy value, and the fix for the frozen-aug bug

    # ---- persistence ----
    HF_USER: str = "Pranilllllll"
    HF_PREFIX: str = "geonusaf-segformer-b0-legacy"
    PUSH_LAST_EVERY: int = 10
    WANDB_PROJECT: str = "geonusaf-baselines-legacy"
    OUT_DIR: str = "/kaggle/working/out"

    ALL_NAMES: tuple = ("Residential", "Road", "River", "Forest",
                        "UnusedLand", "Agricultural", "Unlabelled")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
assert cfg.SPLIT_MODE in ("legacy_kfold", "random", "block")

# Derived, so the two schemes can never disagree with each other:
cfg.N_CLASSES = 6 if cfg.IGNORE_UNLABELLED else 7
cfg.CLASS_NAMES = cfg.ALL_NAMES[:cfg.N_CLASSES]
# Label written for a black pixel: the ignore sentinel, or class 6 in the legacy scheme.
VOID_LABEL = cfg.IGNORE_INDEX if cfg.IGNORE_UNLABELLED else 6
assert cfg.SELECTION_METRIC in ("val_miou_core6", "val_miou_global", "val_miou_legacy")

# The class scheme is part of the run name: a 6-class head cannot load a 7-class checkpoint,
# so the two schemes must never share a repo.
SCHEME_TAG = "c6" if cfg.IGNORE_UNLABELLED else "c7"
RUN_NAME = f"{cfg.HF_PREFIX}-{SCHEME_TAG}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"legacy recipe: lr={cfg.LR} flat | epochs={cfg.MAX_EPOCHS} | patience={cfg.PATIENCE} | "
      f"weights={cfg.CLASS_WEIGHT_MODE} | no sched/clip/EMA/Lovasz")
print(f"scoring: {cfg.N_CLASSES} classes "
      f"({'unlabelled IGNORED' if cfg.IGNORE_UNLABELLED else 'unlabelled is class 6'}) | "
      f"val transform: {'train (legacy)' if cfg.VAL_USES_TRAIN_TF else 'clean'} | "
      f"select on {cfg.SELECTION_METRIC}")

## 3. Determinism

The pasted script seeds nothing, so it is not reproducible — repeating it would give a different
number each time. Seeding here is a strict improvement and does not touch the update rule.

`use_deterministic_algorithms` is left **off**. It forces a sort-based backward kernel for the
bilinear upsample that costs roughly 6x wall-clock on this workload, and at 300 epochs that is the
difference between a fold fitting in a Kaggle session and not. Reproducibility is therefore
"same seed, same machine, same library versions" rather than bit-exact.

`worker_init_fn` seeds from `torch.initial_seed()`, which PyTorch redraws from the loader's
generator **every epoch**. That is the property the legacy run had by accident and the present
notebook lost: augmentation is reproducible across runs but genuinely fresh across epochs. Seeding
workers from a constant would silently recreate the frozen-augmentation bug in a new disguise.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)

def worker_init_fn(worker_id):
    # torch.initial_seed() differs per epoch AND per worker -> live augmentation, still seeded
    s = (torch.initial_seed() + worker_id) % (2 ** 32)
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print("seeded:", cfg.SEED, "| deterministic algorithms: OFF (6x slowdown otherwise)")

## 4. Filename parsing and base-tile grouping

Convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the `mask`
prefix. The grouping is needed for two things: the non-legacy split modes, and the leakage audit
in §8 that counts how many of this fold's validation images have a mirror sibling in training.

`records` is built by walking `sorted(os.listdir(IMG_DIR))`, which is **the same ordering the
pasted script's `SegDataset` used**. `KFold` indices therefore address the same files in both
implementations — that identity is what makes the fold comparison meaningful, so do not re-sort
`records` anywhere below.

One latent hazard in the pasted script, harmless as it happens: it pairs `sorted(images)[idx]`
with `sorted(masks)[idx]` positionally rather than deriving the mask name from the image name.
That is only correct because `image` and `mask` share every suffix and so sort identically. Here
the pairing is explicit and asserted.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):          # SAME ORDER the legacy KFold indexed
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)
IDX_OF = {r[2]: i for i, r in enumerate(records)}      # image name -> legacy KFold index
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print("first 5 in KFold index order:", [r[2] for r in records[:5]])

## 5. Mask colour diagnostic and the class mapping

Same colour table and the same channel-order guard as the reference notebook. The one difference
is what happens to pure black.

The reference notebook maps `(0,0,0)` to `ignore_index=255`, drops it from the loss and from every
metric, and trains 6 classes. The pasted script has `num_classes = 7`, no `ignore_index`, and a
`class_counts` loop that runs over `range(7)` — so its seventh class is the unlabelled region, and
it was **trained on and scored on**. To reproduce its numbers that has to be reproduced too, so the
LUT below sends black to class 6.

`IGNORE_UNLABELLED = True` (the default) keeps the sentinel at 255 instead, excluding those pixels
from the loss and from every metric — the same scheme as the rest of your benchmark. Set it to
`False` to restore the legacy behaviour.

Either way `metrics_from_cm` reports `mIoU_core6` over the six real classes, so the column you
quote in the paper means the same thing in both modes.

In [ ]:
CORE_CLASSES = list(range(6))          # the six real land-use classes, always

BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}

def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = unlabelled sentinel
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = LUT[pack_bgr(m)]
    class_px += np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag = (f"-> class {BGR_TO_CLASS[c]} ({cfg.ALL_NAMES[BGR_TO_CLASS[c]]})" if c in known
           else f"-> {VOID_LABEL} (Unlabelled)")
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(cfg.ALL_NAMES):
    lbl = str(VOID_LABEL) if i == 6 else str(i)
    tail = "  <- excluded from loss and metrics" if (i == 6 and cfg.IGNORE_UNLABELLED) else ""
    print(f"  {lbl:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%{tail}")

assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix.")
print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cache

Unchanged from the reference notebook, and byte-identical to the cache the other baselines use —
if you already saved it as a Kaggle dataset, point `PREBUILT_CACHE` at it and this cell is a no-op.

Labels are written with the `255` sentinel, exactly as before; the remap to class 6 happens at
load time in §7 so the cache stays shareable with the 6-class notebooks.

**This is the one deviation from the pasted script that could not be removed cheaply.** The legacy
`SegDataset` opened the 2345x2351 PNG and let `A.Resize` do the downsample, which is
`cv2.INTER_LINEAR` for the image (masks in albumentations always use nearest, so labels match).
The cache uses `INTER_AREA`, the correct kernel for a 4.6x reduction — bilinear point-samples at
that ratio and aliases thin roads. `INTER_AREA` is strictly the better input, so if this notebook
scores *below* the legacy run it is not because of this line. To close the gap anyway, set
`CACHE_INTERP = cv2.INTER_LINEAR` and a different `CACHE_DIR`, and rebuild (~15-20 min).

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE_INTERP = cv2.INTER_AREA          # legacy used INTER_LINEAR via A.Resize; see the note above
CACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=CACHE_INTERP)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    lab = LUT[pack_bgr(msk)]                                    # full-res colour -> class index
    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts = {}
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist()
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  "
                      f"eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}
sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e9
print(f"cache size: {sz:.2f} GB | per-image class counts for {len(PIX_COUNTS)} files "
      f"(index 6 = unlabelled)")

## 7. Load the cache into RAM

1224 x (512x512x3 + 512x512) uint8 = ~1.3 GB. The `255 -> 6` remap is applied once here rather
than per `__getitem__`, so the training loop never sees the sentinel and `CrossEntropyLoss` needs
no `ignore_index` — matching the pasted script, where every pixel carries a real label.

The assert is the guard that matters most in this notebook: a stray 255 reaching a 7-way loss is
a device-side assert thousands of steps into training, not a clean error.

In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        if not cfg.IGNORE_UNLABELLED:
            lab = np.where(lab == 255, 6, lab).astype(np.uint8)      # legacy 7-class remap
        RAM[iname] = (img, lab)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))
    allowed = set(range(cfg.N_CLASSES)) | ({cfg.IGNORE_INDEX} if cfg.IGNORE_UNLABELLED else set())
    assert set(u.tolist()) <= allowed, f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

## 8. Splitters — legacy KFold over images, plus the grouped controls

### `legacy_kfold` (default)
`KFold(n_splits=5, shuffle=True, random_state=42).split(dataset)` over the 1224 **image indices**,
taking the `FOLD`-th split. `records` is in the same order the legacy `SegDataset` used, so the
indices address the same files.

### The leakage audit
The cell prints how many validation images have at least one of their own flip variants in the
training set. Under `legacy_kfold` expect ~96% — a held-out image whose left-right mirror was a
training example is not a held-out image. Under `random` and `block` it must print exactly 0; the
assert enforces that, so a future edit cannot reintroduce the problem silently.

Read the printed number as the single most important line in this notebook. Whatever mIoU comes
out the other end has to be interpreted against it.

In [ ]:
from sklearn.model_selection import KFold

def legacy_kfold_split(n_items: int, fold: int, n_folds: int, seed: int):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for i, (tr, va) in enumerate(kf.split(np.arange(n_items))):
        if i == fold:
            return list(tr), list(va)
    raise IndexError(fold)

def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

def random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):
    ids = list(base_ids)
    random.Random(seed).shuffle(ids)
    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    val = ids[edges[fold]:edges[fold + 1]]
    train = ids[:edges[fold]] + ids[edges[fold + 1]:]
    return sorted(train), sorted(val)

def expand(ids): return [r for i in ids for r in groups[i]]

if cfg.SPLIT_MODE == "legacy_kfold":
    tr_i, va_i = legacy_kfold_split(len(records), cfg.FOLD, cfg.N_FOLDS, cfg.KFOLD_SEED)
    tr_recs, va_recs = [records[i] for i in tr_i], [records[i] for i in va_i]
elif cfg.SPLIT_MODE == "block":
    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)
    tr_recs, va_recs = expand(tr_ids), expand(va_ids)
else:
    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)
    tr_recs, va_recs = expand(tr_ids), expand(va_ids)

tr_bases, va_bases = {r[0] for r in tr_recs}, {r[0] for r in va_recs}
tr_names = {r[2] for r in tr_recs}
shared_tiles = tr_bases & va_bases
leaky = [r for r in va_recs if any(s[2] in tr_names for s in groups[r[0]] if s[2] != r[2])]

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}/{cfg.N_FOLDS}: "
      f"{len(tr_recs)} train images / {len(va_recs)} val images")
print(f"  base tiles: {len(tr_bases)} train / {len(va_bases)} val / {len(shared_tiles)} in BOTH")
print(f"  VARIANT LEAKAGE: {len(leaky)}/{len(va_recs)} val images "
      f"({len(leaky)/max(len(va_recs),1)*100:.1f}%) have a mirror of themselves in train")

if cfg.SPLIT_MODE != "legacy_kfold":
    assert not shared_tiles and not leaky, "grouped split must not leak variants"
    print("  grouped split: no variant leakage (as required)")

## 9. Class weights — legacy inverse frequency over the whole dataset

The pasted script does this:

```
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * num_classes
```

with `class_counts` accumulated by iterating **`dataset`**, not `Subset(dataset, train_idx)`. Two
consequences, both reproduced here under `WEIGHT_SOURCE = "all"`:

1. Validation class priors enter the loss. A small leak, but a real one, and worth knowing about
   before a reviewer finds it.
2. The weights are raw inverse frequency, uncapped. Over 7 classes the spread is ~153:1; over the
   6 real classes it is 93:1, so River's gradient is amplified about ninety-fold relative to
   Residential.

**This is not meaningfully different from the reference notebook's effective-number weights.** At
pixel-scale counts `1 - beta^n` collapses to `n/N` for every class but the dominant one, so
effective number *becomes* inverse frequency: the two schemes agree within 5% on Road, River,
Forest, UnusedLand and Agricultural, and differ only on Residential (legacy 0.042 vs 0.054). The
reference notebook's `WEIGHT_CAP = 10` never binds — the spread stops at 71.6 by itself. The cell
below prints your actual weights so you can confirm this rather than believe it.

Set `CLASS_WEIGHT_MODE = "none"` for the clean test of the speckle hypothesis: same recipe, same
split, weights off, one fold. If the visual quality improves and mIoU does not collapse, you have
an ablation row rather than a hunch.

In [ ]:
def counts_over(recs):
    n = np.zeros(cfg.N_CLASSES, dtype=np.float64)
    for _, _, iname, _ in recs:
        n += PIX_COUNTS[iname][:cfg.N_CLASSES]
    return n

def legacy_inv_freq_weights(n):
    w = 1.0 / (n + 1e-6)
    return w / w.sum() * cfg.N_CLASSES          # literal transcription

def effective_number_weights(n, cap):
    beta = 1.0 - 1.0 / n.sum()
    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    if cap: w = np.minimum(w, cap)
    return w / w.mean()

SRC = records if cfg.WEIGHT_SOURCE == "all" else tr_recs
COUNTS = counts_over(SRC)

if cfg.CLASS_WEIGHT_MODE == "legacy_inv_freq":
    CLASS_W = legacy_inv_freq_weights(COUNTS)
elif cfg.CLASS_WEIGHT_MODE == "effective_number":
    CLASS_W = effective_number_weights(COUNTS, cfg.EN_BETA_CAP)
elif cfg.CLASS_WEIGHT_MODE == "none":
    CLASS_W = np.ones(cfg.N_CLASSES, dtype=np.float64)
else:
    raise ValueError(cfg.CLASS_WEIGHT_MODE)

print(f"mode: {cfg.CLASS_WEIGHT_MODE} | counted over: {cfg.WEIGHT_SOURCE} "
      f"({len(SRC)} images, {COUNTS.sum():.3e} px)")
print(f"{'class':<14}{'px %':>10}{'weight':>12}")
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}{COUNTS[i]/COUNTS.sum()*100:9.3f}%{CLASS_W[i]:12.4f}")
print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")

CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 10. Dataset and DataLoaders — the legacy albumentations pipeline

The pasted transform, verbatim:

```
A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
           A.RandomRotate90(p=0.5), A.Resize(512, 512), ToTensorV2()])
```

Three things about it are worth stating plainly rather than quietly fixing, because they are part
of what produced the 0.63:

- **The flips are applied on top of flips that are already in the data.** `imageNhf.png` and
  `imageNvf.png` are materialised mirrors of `imageN.png`, so a random hflip at load time can turn
  one file into an exact copy of another file that is sitting in the same epoch. The reference
  notebook drops them for exactly this reason and augments with rot90 only. Reproduced as-is here.
- **The same transform ran on validation.** `Subset(dataset, val_idx)` shares the one
  `SegDataset`, so val images were randomly flipped and rotated at scoring time. In expectation
  that costs little on D4-symmetric nadir imagery, but it makes the val score non-deterministic
  between epochs — which is both why the legacy `val_miou` curve is visibly noisy and why
  checkpoint selection on it is partly a lottery over augmentation draws. `VAL_USES_TRAIN_TF` is
  therefore **`False`** by default here; set it `True` for exact parity. §19 scores through the
  clean loader regardless, so the reported number was never affected — only the choice of which
  epoch to keep.
- **`A.Resize` after the flips is a no-op** on a 512 cache, and albumentations resizes masks with
  nearest regardless, so labels are safe either way.

`num_workers=2`, `persistent_workers=False`, `drop_last=False` are the legacy values. The second
one is not cosmetic: it is what keeps augmentation live across epochs (see §3).

The normalisation path runs the real `SegformerImageProcessor`, as the pasted script does. The
assert below verifies that it is arithmetically identical to `/255` then ImageNet mean/std, so
`PREPROCESS = "manual"` is available as a faster equivalent rather than as an approximation.

In [ ]:
from transformers import SegformerImageProcessor

processor = SegformerImageProcessor.from_pretrained(cfg.HF_MODEL_ID)
PROC_MEAN = np.array(processor.image_mean, np.float32).reshape(3, 1, 1)
PROC_STD  = np.array(processor.image_std,  np.float32).reshape(3, 1, 1)

train_tf = A.Compose([
    A.HorizontalFlip(p=cfg.AUG_HFLIP_P),
    A.VerticalFlip(p=cfg.AUG_VFLIP_P),
    A.RandomRotate90(p=cfg.AUG_ROT90_P),
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    ToTensorV2(),
])
clean_tf = A.Compose([A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE), ToTensorV2()])

class SatSegLegacy(Dataset):
    def __init__(self, recs, tf):
        self.recs, self.tf = recs, tf

    def __len__(self): return len(self.recs)

    def _read(self, iname, mname):
        if cfg.PRELOAD_RAM:
            img, lab = RAM[iname]
            return img.copy(), lab.copy()
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        if not cfg.IGNORE_UNLABELLED:
            lab = np.where(lab == 255, 6, lab).astype(np.uint8)
        return img, lab

    def __getitem__(self, i):
        _, _, iname, mname = self.recs[i]
        img, lab = self._read(iname, mname)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # legacy: PIL .convert("RGB")
        out = self.tf(image=img, mask=lab)
        x, y = out["image"], out["mask"].long()               # x is uint8 CHW
        if cfg.PREPROCESS == "processor":
            x = processor(images=x, return_tensors="pt")["pixel_values"].squeeze(0)
        else:
            x = (x.float().numpy() / 255.0 - PROC_MEAN) / PROC_STD
            x = torch.from_numpy(x)
        return x, y

train_ds = SatSegLegacy(tr_recs, train_tf)
val_ds   = SatSegLegacy(va_recs, train_tf if cfg.VAL_USES_TRAIN_TF else clean_tf)
clean_ds = SatSegLegacy(va_recs, clean_tf)

train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=cfg.DROP_LAST,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=cfg.PERSISTENT_WORKERS,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=cfg.PERSISTENT_WORKERS, worker_init_fn=worker_init_fn)
clean_ld = DataLoader(clean_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True)

# processor vs manual: verify they are the same arithmetic, not merely similar
_img0 = (RAM[tr_recs[0][2]][0] if cfg.PRELOAD_RAM
         else cv2.imread(str(CACHE / "images" / tr_recs[0][2]), cv2.IMREAD_COLOR))
_raw = torch.from_numpy(np.ascontiguousarray(
    cv2.cvtColor(_img0, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)))
_a = processor(images=_raw, return_tensors="pt")["pixel_values"].squeeze(0).numpy()
_b = (_raw.float().numpy() / 255.0 - PROC_MEAN) / PROC_STD
assert np.abs(_a - _b).max() < 1e-4, np.abs(_a - _b).max()
print(f"processor == manual normalisation (max abs diff {np.abs(_a-_b).max():.2e})")

xb, yb = next(iter(train_ld))
print(xb.shape, xb.dtype, yb.shape, yb.dtype, "| label values:", torch.unique(yb).tolist())
print(f"steps/epoch: {len(train_ld)} | val batches: {len(val_ld)} | "
      f"persistent_workers={cfg.PERSISTENT_WORKERS} (live augmentation)")

## 11. Loss — plain weighted cross-entropy

`nn.CrossEntropyLoss(weight=class_weights)`. No Lovasz term, no label smoothing.

`ignore_index=255` is passed so that unlabelled pixels contribute no gradient. In the legacy
7-class scheme no pixel is ever 255, so the argument is inert and the loss is byte-for-byte the
pasted one; under the default 6-class scheme it is what keeps annotation gaps out of training.

This is the single biggest *recipe* difference from the reference notebook, which adds
`0.75 * Lovasz-Softmax`. Lovasz is the convex surrogate for Jaccard and normally **raises** mIoU on
thin classes, so its absence here is not a plausible explanation for the legacy run scoring
higher — but it does mean the two runs optimise different objectives, and the comparison in §25
is what establishes which one actually wins on a matched split and a matched metric.

The loss is computed **inside** `autocast`, as in the pasted script. In practice this changes
nothing: `cross_entropy` is on PyTorch's fp32 autocast list and is promoted automatically.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=CLASS_W_T, ignore_index=cfg.IGNORE_INDEX)
print(f"loss: CrossEntropyLoss(weight=class_weights, ignore_index={cfg.IGNORE_INDEX}) "
      f"| no Lovasz | no smoothing")
print(f"  ignore_index is inert in the legacy 7-class scheme (no pixel is ever 255 there)")
print("weights:", dict(zip(cfg.CLASS_NAMES, np.round(CLASS_W, 4).tolist())))

## 12. Model, optimizer — and everything the legacy script did not have

```
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=7, ignore_mismatched_sizes=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
```

That is the whole of it. No `drop_path_rate`, no `classifier_dropout_prob` — the checkpoint's own
config values apply. **One parameter group**, so the MiT encoder and the decode head both train at
3e-5; there is no 10x differential. `weight_decay` is torch's AdamW default of 0.01, applied to
every tensor including LayerNorm gains and biases, which is not what you would choose but is what
ran. No scheduler at all: the LR is 3e-5 at step 0 and 3e-5 at step 18,000.

### The resolution trap, unchanged
SegFormer emits logits at H/4 = 128x128. The pasted script upsamples them to the label size with
`F.interpolate(..., mode="bilinear", align_corners=False)` before the loss, which is the
mmsegmentation convention and the same thing `forward_logits` does in the reference notebook. The
probe assert below is the one check you should never skip: if it prints 128, every metric in this
notebook is computed at the wrong resolution.

### What flat 3e-5 for 300 epochs buys
With ~62 steps/epoch, warmup-free training at 3e-5 on an ADE20K-pretrained MiT-B0 is stable — your
`train_loss` curve descends smoothly with no spike, which is the evidence that no warmup was
needed. The absence of a cosine decay means the model never gets the end-of-schedule annealing
bump, but it also never anneals into a minimum early, and at 300 epochs that trade appears to have
favoured the long flat run: `val_miou` is still rising at the right edge of your curve.

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

def build_model():
    return SegformerForSemanticSegmentation.from_pretrained(
        cfg.HF_MODEL_ID,
        num_labels=cfg.N_CLASSES,
        ignore_mismatched_sizes=True,      # ADE20K head is 150-way; ours is 7-way
    )                                      # NOTE: no dropout overrides -- legacy passed none

def forward_logits(m, x):
    lo = m(pixel_values=x).logits
    return F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)

model = build_model().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR)      # single group, flat LR
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)
scheduler = None                                                   # legacy: none

n_par = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    probe = forward_logits(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))
assert probe.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe.shape
assert probe.shape[1] == cfg.N_CLASSES, probe.shape
print(f"params: {n_par/1e6:.2f} M | logits after upsample: {tuple(probe.shape)}")
print(f"param groups: {len(optimizer.param_groups)} | lr: {cfg.LR} (flat) | "
      f"wd: {optimizer.param_groups[0]['weight_decay']} | scheduler: {cfg.SCHEDULER}")
print(f"drop_path_rate={model.config.drop_path_rate} "
      f"classifier_dropout={model.config.classifier_dropout_prob} (checkpoint defaults)")

## 13. Metrics — the legacy number and the pooled number, side by side

`legacy_batch_miou` is a literal transcription of the pasted `mean_iou`. Read what it does:

```
for c in range(num_classes):
    inter = ((preds == c) & (targets == c)).sum()
    union = ((preds == c) | (targets == c)).sum()
    if union > 0: ious.append(inter / union)
return mean(ious)
```

IoU is formed **within one batch of 16 images**, classes whose batch union is zero are dropped,
and the epoch score is the unweighted mean over batches. That is a mean of ratios; the confusion
matrix gives a ratio of means. Different estimators of the same quantity, and they do not agree.

Do not assume they disagree in your favour, though. Simulated at your class shares and tile
presence rates, the difference lands within about ±0.01 and its sign flips depending on how
rare-class area is distributed across batches — a batch holding fifty River pixels that the model
misses contributes a hard 0.0 to the average, which pulls it *down*, while pooling would barely
notice those pixels. Union-0 skipping is likewise rarer than it sounds: a batch of 16 tiles
contains River about 97% of the time. The batch estimator is wrong, but it is probably not where
most of your 0.17 went. Measure it rather than assume it — which is what the three series below
are for.

Both are computed every epoch here. **Early stopping and checkpoint selection use the legacy one**,
because changing the selection rule would change which weights get saved and this notebook is
meant to reproduce the run, not improve it. The logged series are:

| key | what it is |
|---|---|
| `val_miou_legacy` | the pasted estimator: batch-averaged, over the scored classes |
| `val_miou_global` | pooled from the confusion matrix, over the scored classes |
| `val_miou_core6` | pooled, 6 real classes — **the number comparable to your benchmark table** |

Under `IGNORE_UNLABELLED = True` the last two coincide, because the scored classes *are* the six
real ones; the first still differs, and the difference is the estimator alone.

**`best.pt` is selected on `cfg.SELECTION_METRIC`, default `val_miou_core6`.** The pasted script
selected on the batch estimator, which is both the noisiest of the three and — in your fold-1 run
— about 0.05 below the pooled one. Selecting a checkpoint on a noisy criterion that is not the one
you report is a real, if small, way to lose mIoU: you keep whichever epoch got a lucky batch
partition rather than whichever epoch was actually best. Set it back to `"val_miou_legacy"` only
if you are chasing exact parity.

In [ ]:
from skimage.morphology import skeletonize

def legacy_batch_miou(logits, targets):
    preds = logits.argmax(dim=1)
    k = targets != cfg.IGNORE_INDEX          # inert in the 7-class scheme
    preds, targets = preds[k], targets[k]
    ious = []
    for c in range(cfg.N_CLASSES):
        inter = ((preds == c) & (targets == c)).sum().float()
        union = ((preds == c) | (targets == c)).sum().float()
        if union > 0:
            ious.append(inter / union)
    if len(ious) == 0:
        return 0.0
    return float(torch.mean(torch.stack(ious)))

class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX          # MUST come first: 255 would overflow the bincount
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.cpu().numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall
    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {
        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),
        "mIoU_core6": float(np.nanmean(iou[CORE_CLASSES])),
        "mF1_core6": float(np.nanmean(f1[CORE_CLASSES])),
        "OA": float(po), "FWIoU": float((freq * iou).sum()),
        "kappa": float((po - pe) / max(1 - pe, 1e-12)),
        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},
        "confusion_matrix_raw": cm.astype(np.int64).tolist(),
        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),
    }

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

print(f"metrics ready | {cfg.N_CLASSES} scored classes | selection metric: {cfg.SELECTION_METRIC}")

## 14. Efficiency metrics

Unchanged from the reference notebook so the rows stay comparable: params, GFLOPs at 512x512 via
`ptflops`, peak training VRAM, wall-clock, inference FPS at batch 1 with 20 warmup iterations.

`ptflops` hooks `nn.Module`s and misses the functional attention matmuls, so transformer rows are
slightly under-counted against CNN rows — say so in the caption, or re-measure the whole table
with one tool. Published reference: SegFormer-B0 is ~3.8 M params / ~8.4 GFLOPs at 512x512. The
7-way head adds a handful of parameters over the 6-way one, which is why this row will not match
the reference notebook's to the last digit.

In [ ]:
class _FlopsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return forward_logits(self.m, x)

def efficiency_report(m):
    from ptflops import get_model_complexity_info
    m.eval()
    w = _FlopsWrapper(m)
    with torch.cuda.amp.autocast(enabled=False):
        macs, params = get_model_complexity_info(
            w, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),
            as_strings=False, print_per_layer_stat=False, verbose=False)
    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(20): forward_logits(m, x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): forward_logits(m, x)
        torch.cuda.synchronize()
    return {"params_M": params / 1e6,
            "GFLOPs_512": macs * 2 / 1e9,
            "inference_fps_bs1": 50 / (time.time() - t0)}

## 15. Hugging Face persistence and resume

One repo per fold, `geonusaf-segformer-b0-legacy-{mode}-fold{k}`, so nothing overwrites anything
and §21 can pull all five back. The prefix carries `legacy` deliberately: these checkpoints must
never be confused with the protocol runs when the results table is assembled.

- `best.pt` — weights + config + metrics, pushed whenever the selection metric improves.
- `last.pt` — full state including optimizer and scaler, pushed every `PUSH_LAST_EVERY` epochs.
  300 epochs will not fit one Kaggle session; this is what makes the fold resumable.

The model card records the legacy settings explicitly, including the variant-leakage percentage
from §8. A card that reports 0.63 without that number next to it is a card that will eventually
mislead someone — most likely you, six weeks from now.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg):
    api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                    repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)

def hf_try_download(name):
    try: return hf_hub_download(HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

LEAK_PCT = len(leaky) / max(len(va_recs), 1) * 100

def push_model_card(best_metrics, epoch, legacy_miou):
    pc = best_metrics.get("per_class", {})
    L = ["---",
         "tags: [semantic-segmentation, remote-sensing, segformer, mit-b0, geonusaf, legacy-parity]",
         "library_name: transformers", "---", "",
         f"# GeoNUSAF - SegFormer-B0 LEGACY PARITY - {cfg.SPLIT_MODE}, fold {cfg.FOLD}", "",
         "Pre-protocol training recipe (flat AdamW 3e-5, 300 epochs, plain weighted CE, no",
         "scheduler/clipping/EMA). Read the split, class-scheme and metric fields below before",
         "putting any of these numbers next to a protocol run.", "",
         "| field | value |", "|---|---|",
         f"| architecture | SegformerForSemanticSegmentation, {cfg.HF_MODEL_ID} |",
         f"| split mode | {cfg.SPLIT_MODE} ({cfg.N_FOLDS}-fold, seed {cfg.KFOLD_SEED}) |",
         f"| fold | {cfg.FOLD} |",
         f"| VARIANT LEAKAGE | {len(leaky)}/{len(va_recs)} val images ({LEAK_PCT:.1f}%) have a mirror in train |",
         f"| classes | {cfg.N_CLASSES} ({', '.join(cfg.CLASS_NAMES)}) |",
         f"| unlabelled pixels | {'ignored (index 255)' if cfg.IGNORE_UNLABELLED else 'trained as class 6'} |",
         f"| class weights | {cfg.CLASS_WEIGHT_MODE}, counted over {cfg.WEIGHT_SOURCE}, cap {cfg.WEIGHT_CAP} |",
         f"| loss | weighted CE only (lovasz {cfg.LAMBDA_LOVASZ}, smoothing {cfg.LABEL_SMOOTHING}) |",
         f"| optimizer | AdamW single group, lr {cfg.LR} flat, wd {cfg.WEIGHT_DECAY} |",
         f"| scheduler / clip / EMA | {cfg.SCHEDULER} / {cfg.CLIP_NORM} / {cfg.USE_EMA} |",
         f"| epochs / patience / batch | {cfg.MAX_EPOCHS} / {cfg.PATIENCE} / {cfg.BATCH_SIZE} |",
         f"| val transform | {'train transform (legacy)' if cfg.VAL_USES_TRAIN_TF else 'clean'} |",
         f"| selection metric | {cfg.SELECTION_METRIC} |",
         f"| best epoch | {epoch} |",
         f"| val mIoU (batch-averaged estimator) | {legacy_miou:.4f} |",
         f"| val mIoU (pooled, all scored classes) | {best_metrics.get('mIoU', float('nan')):.4f} |",
         f"| val mIoU (pooled, 6 real classes) | {best_metrics.get('mIoU_core6', float('nan')):.4f} |",
         f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |",
         f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |", "",
         "## Per-class (validation, pooled)", "| class | IoU | F1 |", "|---|---|---|"]
    for k, v in pc.items():
        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write("\n".join(L))
    hf_push(p, "README.md", f"card epoch {epoch}")

print("HF ready:", HF_REPO, f"| leakage on this fold: {LEAK_PCT:.1f}%")

## 16. Weights & Biases

One behavioural fix relative to the pasted script, and it is the reason your curves are hard to
read: that script calls `wandb.log(...)` once per epoch **and** again inside `log_images_wandb`
every 5 epochs, with no `step=` argument. W&B advances its own step counter on each call, so 300
epochs render as ~360 steps and the x-axis silently stops being the epoch number. That is why
fold-2 appears to run past 300 — it did not; it ran the full 300 epochs without ever early
stopping, which is itself the more interesting reading of that plot.

Here every log passes `step=epoch`, so the axis means what it says. All three mIoU variants are
logged, plus `overfit_gap` and both grad-norm series.

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config={**asdict(cfg), "variant_leakage_pct": LEAK_PCT,
                                       "train_images": len(tr_recs), "val_images": len(va_recs)})

## 17. Train / validate loops

The update is exactly the pasted one: forward under `autocast`, upsample logits, weighted CE,
`scaler.scale(loss).backward()`, `scaler.step`, `scaler.update()`. **No `unscale_`, no clipping,
no scheduler step, no EMA.**

### The grad-norm curve
The pasted script computes its gradient norm straight after `scaler.scale(loss).backward()`, over
gradients that are still multiplied by the loss scale — typically 2^16. That is why the plot you
attached reads 100,000 to 600,000 rather than 1 to 10, and why it looks like an explosion that
never destabilised anything. It is a logging artefact, not a training event.

Both series are logged here. `grad_norm_scaled` reproduces your existing curve so the two runs
overlay; `grad_norm_true` is that value divided by the current loss scale, which is the real
number. Dividing is exact — unscaling is a uniform division — so this costs nothing and does not
touch the update.

Validation runs the model in `eval()` over the val loader (which carries the train transform, per
§10) and accumulates both estimators in one pass.

In [ ]:
def train_one_epoch(epoch):
    model.train()
    tot, n = 0.0, 0
    gn_scaled, gn_true = [], []
    for it, (x, y) in enumerate(train_ld):
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(model, x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()

        # legacy grad-norm probe: computed on SCALED grads, exactly as the pasted script did
        sq = 0.0
        for p in model.parameters():
            if p.grad is not None:
                sq += float(p.grad.data.norm(2)) ** 2
        gs = sq ** 0.5
        gn_scaled.append(gs)
        gn_true.append(gs / max(scaler.get_scale(), 1e-12) if cfg.AMP else gs)

        scaler.step(optimizer)          # no unscale_, no clip_grad_norm_
        scaler.update()

        tot += float(loss) * x.size(0); n += x.size(0)
        if it % 20 == 0:
            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss):.4f} "
                  f"gnorm_true {gn_true[-1]:.3f} (scaled {gs:.0f})", flush=True)
    return (tot / max(n, 1), float(np.mean(gn_scaled)),
            float(np.mean(gn_true)), float(np.max(gn_true)))

@torch.no_grad()
def validate(loader=None, m=None):
    loader = loader if loader is not None else val_ld
    m = m if m is not None else model
    m.eval()
    cm = ConfMat(cfg.N_CLASSES)
    tot, n, mious = 0.0, 0, []
    for x, y in loader:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(m, x)
            loss = criterion(logits, y)
        lf = logits.float()
        mious.append(legacy_batch_miou(lf, y))          # legacy estimator, per batch
        cm.update(lf.argmax(1), y)                      # pooled estimator
        tot += float(loss) * x.size(0); n += x.size(0)
    return tot / max(n, 1), float(np.mean(mious)), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)

## 18. Main loop with automatic resume

Early stopping is the pasted logic transcribed, including its shape: the patience counter resets
on any improvement, and the `break` lives in the `else` branch.

```
if val_miou > best_miou:  best_miou = val_miou; patience_counter = 0; save
else:                     patience_counter += 1
                          if patience_counter >= patience: break
```

**Budget.** ~980 train images at batch 16 is ~62 steps/epoch, ~245 val images is 16 batches. The
per-step gradient-norm probe walks every parameter tensor and costs real time at this model size —
it is kept because it is what produced your curves, but it is the first thing to delete if you are
short on session hours. Expect roughly 45-75 s/epoch, so **300 epochs is 4-6 h per fold and will
not finish in one 12 h session together with everything else**. `last.pt` resume is not optional
here; plan on two sessions per fold, five folds.

If the session dies mid-fold, rerun the notebook unchanged and it continues from `last.pt`. Note
that RNG state is not restored, so a resumed fold is reproducible only up to the resume point.

In [ ]:
ck = hf_try_download("last.pt")
start_epoch, best_miou, best_epoch, bad, history = 0, 0.0, -1, 0, []

if ck:
    # Belt and braces: the repo name already separates the schemes, but a checkpoint whose head
    # is the wrong width must never reach load_state_dict -- it fails hours into a session.
    _st = torch.load(ck, map_location="cpu")
    _nc = _st.get("n_classes",
                  _st["model_state"]["decode_head.classifier.weight"].shape[0])
    if _nc != cfg.N_CLASSES:
        print(f"IGNORING last.pt: it has a {_nc}-class head, this run has {cfg.N_CLASSES}. "
              f"Starting fresh. (Check HF_REPO -- it should end in -{SCHEME_TAG}-.)")
        ck = None
    del _st

if ck:
    st = torch.load(ck, map_location="cuda")
    model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optim_state"])
    scaler.load_state_dict(st["scaler_state"])
    start_epoch = st["epoch"] + 1; best_miou = st["best_miou"]; best_epoch = st["best_epoch"]
    bad = st["bad"]; history = st["history"]
    print(f"RESUMED from epoch {st['epoch']} "
          f"(best {cfg.SELECTION_METRIC} {best_miou:.4f} @ e{best_epoch})")
else:
    print("fresh start")

torch.cuda.reset_peak_memory_stats()
t_start, best_metrics, best_legacy = time.time(), {}, 0.0

for epoch in range(start_epoch, cfg.MAX_EPOCHS):
    te = time.time()
    tr_loss, gn_s, gn_t, gn_tmax = train_one_epoch(epoch)
    va_loss, va_legacy, m = validate()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
           "overfit_gap": va_loss - tr_loss,
           "val_miou_legacy": va_legacy,            # the pasted estimator -- selection metric
           "val_miou_global": m["mIoU"],            # pooled, 7 classes
           "val_miou_core6": m["mIoU_core6"],       # pooled, 6 real classes
           "val_mF1": m["mF1"], "val_OA": m["OA"], "val_kappa": m["kappa"],
           "grad_norm_scaled": gn_s, "grad_norm_true": gn_t, "grad_norm_true_max": gn_tmax,
           "lr": optimizer.param_groups[0]["lr"], "peak_vram_gb": peak_gb,
           "epoch_sec": time.time() - te,
           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}
    history.append(row)
    if WANDB: wandb.log(row, step=epoch)
    print(f"[e{epoch}] train {tr_loss:.4f} | val {va_loss:.4f} | "
          f"mIoU legacy {va_legacy:.4f} / pooled {m['mIoU']:.4f} / core6 {m['mIoU_core6']:.4f} | "
          f"gnorm {gn_t:.2f} | {row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB")

    sel = row[cfg.SELECTION_METRIC]
    if sel > best_miou:
        best_miou, best_epoch, bad = sel, epoch, 0
        best_metrics, best_legacy = m, va_legacy
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": model.state_dict(),
                    "model_config": model.config.to_dict(),
                    "cfg": asdict(cfg), "metrics": m, "legacy_miou": va_legacy,
                    "n_classes": cfg.N_CLASSES, "epoch": epoch}, p)
        hf_push(p, "best.pt", f"best {cfg.SELECTION_METRIC} {best_miou:.4f} @ e{epoch}")
        push_model_card(best_metrics, epoch, best_legacy)
        print(f"  -> new best {best_miou:.4f}, pushed")
    else:
        bad += 1

    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),
                    "scaler_state": scaler.state_dict(), "epoch": epoch,
                    "n_classes": cfg.N_CLASSES,
                    "best_miou": best_miou, "best_epoch": best_epoch,
                    "bad": bad, "history": history}, p)
        hf_push(p, "last.pt", f"state e{epoch}")

    if bad >= cfg.PATIENCE:
        print(f"EARLY STOP at epoch {epoch} (best {best_miou:.4f} @ e{best_epoch})"); break

wall_h = (time.time() - t_start) / 3600
print(f"training done | best {cfg.SELECTION_METRIC} {best_miou:.4f} @ epoch {best_epoch} | "
      f"this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpoint — both protocols

The best checkpoint is scored **twice**, and the difference between the two is the point of this
section:

1. **Legacy protocol** — through `val_ld`, which carries the train transform, using the per-batch
   estimator over 7 classes. This is the number that is directly comparable to your 0.63.
2. **Clean protocol** — through `clean_ld` (resize only, no random flips, deterministic), pooled
   from a confusion matrix, reported over all 7 classes and over the 6 real ones. This is the
   number that is comparable to the 0.4765 / 0.3962 rows in your benchmark table.

clDice for Road and River and the efficiency block run on the clean pass only. clDice averages
over images where the class is present in ground truth or prediction; images where it is absent
from both are skipped rather than scored 1.0, which would inflate River badly.

If the two numbers are far apart — and they will be — the gap is not noise and it is not
architecture. It is the estimator. Report the clean one and mention the legacy one in a footnote
if you want to explain the earlier figure at all.

In [ ]:
# On a resumed session best.pt may exist only on the Hub: this session may never have improved.
_bp = f"{cfg.OUT_DIR}/best.pt"
if not os.path.exists(_bp):
    _bp = hf_try_download("best.pt")
    assert _bp, "no best.pt locally or on the Hub -- did any epoch ever improve?"
    print("best.pt pulled from the Hub (no improvement in this session)")
_ck = torch.load(_bp, map_location="cuda")
_nc = _ck.get("n_classes", _ck["model_state"]["decode_head.classifier.weight"].shape[0])
assert _nc == cfg.N_CLASSES, (
    f"best.pt has a {_nc}-class head but this run is {cfg.N_CLASSES}-class -- wrong repo")
best_model = build_model().cuda()
best_model.load_state_dict(_ck["model_state"])
best_model.eval()

# ---- 1. legacy protocol: train-transform loader + per-batch estimator ----
_, legacy_miou_final, legacy_pooled = validate(loader=val_ld, m=best_model)

# ---- 2. clean protocol: deterministic loader, pooled confusion matrix, clDice ----
cm = ConfMat(cfg.N_CLASSES)
cld = {1: [], 2: []}                                   # Road, River
with torch.no_grad():
    for x, y in clean_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(best_model, x)
        pred = logits.float().argmax(1)
        cm.update(pred, y)
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        valid = yn != VOID_LABEL                       # do not score clDice inside annotation gaps
        for b in range(pn.shape[0]):
            for c in cld:
                v = cldice_pair((pn[b] == c) & valid[b], (yn[b] == c) & valid[b])
                if v is not None: cld[c].append(v)

final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)
final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}
final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}
final["miou_batch_estimator"] = legacy_miou_final
final["miou_pooled_valloader"] = legacy_pooled["mIoU"]
final["variant_leakage_pct"] = LEAK_PCT
final.update(efficiency_report(best_model))
final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
final["train_wall_clock_h_this_session"] = wall_h
final["best_epoch"] = best_epoch
final["run"] = {"model": "SegFormer-B0 (legacy recipe)", "split_mode": cfg.SPLIT_MODE,
                "fold": cfg.FOLD, "seed": cfg.SEED, "n_classes": cfg.N_CLASSES,
                "ignore_unlabelled": cfg.IGNORE_UNLABELLED,
                "selection_metric": cfg.SELECTION_METRIC,
                "class_weight_mode": cfg.CLASS_WEIGHT_MODE}

NC = cfg.N_CLASSES
print("=" * 72)
print(f"batch-averaged estimator, val loader, {NC} classes : {legacy_miou_final:.4f}")
print(f"pooled,                   val loader, {NC} classes : {legacy_pooled['mIoU']:.4f}")
print(f"pooled,                   clean loader, {NC} cls   : {final['mIoU']:.4f}")
print(f"pooled,                   clean loader, 6 real cls : {final['mIoU_core6']:.4f}"
      f"   <- the number for the paper")
print(f"variant leakage on this fold                       : {LEAK_PCT:.1f}%")
print("=" * 72)
print(f"OA {final['OA']:.4f} | FWIoU {final['FWIoU']:.4f} | kappa {final['kappa']:.4f} | "
      f"mF1(6) {final['mF1_core6']:.4f}")
print(f"clDice Road {final['clDice']['Road']} | River {final['clDice']['River']}")
print("\nper class (clean, pooled):")
for k, v in final["per_class"].items():
    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  "
          f"PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}")

## 20. Persist results

`results_{mode}_fold{k}.json` (full, including confusion matrices), a flat `.csv` row for the
comparison table, and `history_{mode}_fold{k}.csv` for the curves, all pushed to the fold's repo.

The flat row carries `split_mode`, `n_classes`, `variant_leakage_pct`, `class_weight_mode` and the
`legacy_*` columns alongside the pooled ones. That is deliberate: the failure mode this whole
exercise exists to prevent is a legacy number and a protocol number ending up in the same table
column with nothing to tell them apart.

In [ ]:
import csv
tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2)

flat = {"model": "SegFormer-B0-legacy", "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD,
        "seed": cfg.SEED, "n_classes": cfg.N_CLASSES,
        "ignore_unlabelled": cfg.IGNORE_UNLABELLED, "selection_metric": cfg.SELECTION_METRIC,
        "val_transform": "train" if cfg.VAL_USES_TRAIN_TF else "clean",
        "variant_leakage_pct": LEAK_PCT, "class_weight_mode": cfg.CLASS_WEIGHT_MODE,
        "lr": cfg.LR, "epochs_max": cfg.MAX_EPOCHS, "best_epoch": final["best_epoch"],
        "miou_batch_estimator": final["miou_batch_estimator"],
        "miou_pooled_valloader": final["miou_pooled_valloader"],
        "mIoU_clean_allclasses": final["mIoU"], "mIoU_clean_core6": final["mIoU_core6"],
        "mF1_core6": final["mF1_core6"], "OA": final["OA"],
        "FWIoU": final["FWIoU"], "kappa": final["kappa"],
        "params_M": final["params_M"], "GFLOPs_512": final["GFLOPs_512"],
        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"]}
for k, v in final["per_class"].items():
    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]
for k, v in final["clDice"].items():
    flat[f"clDice_{k}"] = v

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv")]:
    hf_push(p, n, "results")
if WANDB: wandb.finish()
print("results pushed to", HF_REPO)

## 21. Aggregation across folds — run after all folds finish

Pulls every fold's CSV and reports mean +/- SD for each of the four mIoU definitions side by side.
The table it prints is the answer to your question, in one place:

- **`legacy_miou_batch_7class`** should land near 0.63 if the transcription is faithful. That is
  the check that this notebook really did reproduce the old run.
- **`mIoU_clean_core6`** is what the same weights are worth under the protocol's scoring rules.
- The **difference** between them, at fixed weights and a fixed split, is pure estimator effect —
  no architecture, no recipe, no hyper-parameter.

The second block does the same across `SPLIT_MODE`, so if you have run the decomposition in §25
you get the leakage effect and the geography effect separated out as well.

The third block prints where `val_miou_legacy` peaked per fold. If a fold peaks at its final epoch
it did **not** converge and 300 epochs was a floor, not a ceiling — that is what your fold-2 curve
is showing.

In [ ]:
import pandas as pd
MIOU_COLS = ["miou_batch_estimator", "miou_pooled_valloader",
             "mIoU_clean_allclasses", "mIoU_clean_core6"]

MAX_FOLDS = 5            # scan every fold index any mode could have used, not just this session's
def candidate_repos(mode, k):
    # "" covers runs made before the c6/c7 tag was introduced
    return [f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{t}{mode}-fold{k}" for t in ("c6-", "c7-", "")]

rows = []
for mode in ["legacy_kfold", "random", "block"]:
    for k in range(MAX_FOLDS):
        for repo in candidate_repos(mode, k):
            try:
                rows.append(pd.read_csv(hf_hub_download(repo, f"results_{mode}_fold{k}.csv",
                                                        token=HF_TOKEN)))
            except Exception:
                pass

if rows:
    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"{cfg.OUT_DIR}/aggregate_segformer_b0_legacy.csv", index=False)
    print("--- the four mIoU definitions, same weights, same folds ---")
    print(df.groupby("split_mode")[MIOU_COLS].agg(["mean", "std"]).round(4).to_string())
    print("\n--- leakage present on each split mode ---")
    print(df.groupby("split_mode")["variant_leakage_pct"].mean().round(1).to_string())
    if "legacy_kfold" in set(df.split_mode) and "block" in set(df.split_mode):
        a = df[df.split_mode == "legacy_kfold"].miou_batch_estimator.mean()
        b = df[df.split_mode == "block"].mIoU_clean_core6.mean()
        print(f"\nTOTAL GAP legacy-reported vs protocol-reported: {a:.4f} -> {b:.4f} "
              f"({b-a:+.4f})")
else:
    print("no fold CSVs found yet -- run the folds first")

print("\n--- did each fold converge? ---")
for mode in ["legacy_kfold", "random", "block"]:
    for k in range(MAX_FOLDS):
        h = None
        for repo in candidate_repos(mode, k):
            try:
                h = pd.read_csv(hf_hub_download(repo, f"history_{mode}_fold{k}.csv",
                                                token=HF_TOKEN)); break
            except Exception:
                continue
        if h is None or "val_miou_core6" not in h:
            continue
        em, last = int(h.val_miou_core6.idxmax()), int(h.epoch.max())
        flag = "  <-- STILL IMPROVING AT THE END" if em >= last - 5 else ""
        print(f"  {mode} f{k}: peak core6 mIoU {h.val_miou_core6.max():.4f} @ e{em} "
              f"of {last}{flag}")
if rows:
    print("\nNOTE: rows may mix 6-class and 7-class runs -- check the n_classes column "
          "before comparing any two.")

## 22. Standalone inference from Hugging Face

Self-contained. The model is rebuilt from the HF config embedded in `best.pt`, so it reconstructs
the trained architecture exactly, and the preprocessing matches training: `INTER_AREA` downsample
to 512, `/255`, ImageNet mean/std, H/4 logits upsampled back to 512.

Note the head is **7-way** here. Class 6 is `Unlabelled` and will appear in predictions — when you
render or compare against a 6-class model, decide explicitly whether to treat it as a class or
mask it out, and do the same thing on both sides.

In [ ]:
def load_from_hf(repo_id, token=None, device="cuda"):
    import torch, cv2
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location=device)
    m = SegformerForSemanticSegmentation(SegformerConfig.from_dict(ck["model_config"]))
    m = m.to(device).eval()
    m.load_state_dict(ck["model_state"])
    return m, ck["cfg"], ck["metrics"]

@torch.no_grad()
def predict_paths(model, c, paths, device="cuda", batch=8):
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std  = np.array([0.229, 0.224, 0.225], np.float32)
    out = []
    for i in range(0, len(paths), batch):
        xs = []
        for p in paths[i:i + batch]:
            im = cv2.imread(str(p), cv2.IMREAD_COLOR)
            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)
            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))
        x = torch.stack(xs).to(device)
        with torch.cuda.amp.autocast():
            lo = model(pixel_values=x).logits
            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
            out.append(lo.float().argmax(1).cpu().numpy())
    return np.concatenate(out)

INFER_REPO = HF_REPO
inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)
print(f"loaded {INFER_REPO} | pooled val mIoU {inf_metrics['mIoU']:.4f} "
      f"| core6 {inf_metrics['mIoU_core6']:.4f}")
demo = [IMG_DIR / r[2] for r in va_recs[:4]]
preds = predict_paths(inf_model, inf_cfg, demo)
print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## 23. Qualitative triptychs — image / GT / prediction

Rendered through the **clean** loader, so what you see is the deterministic prediction rather than
a randomly flipped one. Class 6 renders black in both ground truth and prediction; unlike the
6-class notebooks, black in the prediction panel here is a real model output, not a blanked-out
ignore region.

This is the figure to look at for your speckle question. What to check specifically: whether River
and Road predictions are coherent ribbons or scattered pixels, and whether UnusedLand is bleeding
into Agricultural. If heavy inverse-frequency weighting is hurting visual quality, it shows up as
rare-class confetti in otherwise-correct Residential blocks — and the way to confirm it is to
re-run one fold with `CLASS_WEIGHT_MODE = "none"` and put the two figures next to each other,
not to reason about it from one image.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]
PALETTE_RGB[VOID_LABEL] = (0, 0, 0)

def colorize(lab): return PALETTE_RGB[lab]

@torch.no_grad()
def per_image_miou(k=3):
    scores = []
    for j in range(0, len(clean_ds), 16):
        idxs = list(range(j, min(j + 16, len(clean_ds))))
        xs, ys = zip(*[clean_ds[i] for i in idxs])
        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            p = forward_logits(best_model, x).float().argmax(1)
        for b, i in enumerate(idxs):
            c = ConfMat(cfg.N_CLASSES); c.update(p[b], y[b])
            m1 = c.numpy()
            tp = np.diag(m1); un = m1.sum(0) + m1.sum(1) - tp
            sel = [k for k in CORE_CLASSES if un[k] > 0]      # absent classes are not scored 0
            scores.append((float(np.mean(tp[sel] / un[sel])) if sel else 0.0, i))
    scores.sort(key=lambda t: t[0])
    return scores[-k:][::-1], scores[:k]

best_imgs, worst_imgs = per_image_miou(k=3)

def triptych(items, title):
    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))
    axes = np.atleast_2d(axes)
    for r, (score, i) in enumerate(items):
        _, _, iname, mname = clean_ds.recs[i]
        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)
        x, y = clean_ds[i]
        gt = y.numpy().astype(np.uint8)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
            pr = forward_logits(best_model, x[None].cuda()).float().argmax(1)[0]
        pr = pr.cpu().numpy().astype(np.uint8)
        if cfg.IGNORE_UNLABELLED:
            pr = np.where(gt == cfg.IGNORE_INDEX, cfg.IGNORE_INDEX, pr).astype(np.uint8)
        panels = [(img, iname), (colorize(gt), "ground truth"),
                  (colorize(pr), f"prediction (core6 mIoU {score:.3f})")]
        for c, (im, t) in enumerate(panels):
            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]
    handles.append(Patch(color="black", label="unlabelled"
                         + (" (ignored)" if cfg.IGNORE_UNLABELLED else "")))
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{title} - SegFormer-B0 legacy, {cfg.SPLIT_MODE} fold {cfg.FOLD} "
                 f"(weights: {cfg.CLASS_WEIGHT_MODE})", fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
    hf_push(p, os.path.basename(p), "qualitative")

triptych(best_imgs, "best cases")
triptych(worst_imgs, "worst cases")

## 24. Confusion matrix figure

Row-normalised, 7x7. The `Unlabelled` row and column are the ones to read first: they are absent
from every 6-class confusion matrix you have, and they show how much of this model's apparent
competence is spent on annotation gaps.

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])
N = cfg.N_CLASSES
fig, ax = plt.subplots(figsize=(8.0, 6.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - SegFormer-B0 legacy, {cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = f"{cfg.OUT_DIR}/confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist, deviations, and the experiment that settles this

### Checklist
1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. §5 must print **channel-order check PASSED**.
3. §8 prints the variant-leakage percentage. Write it down. Under `legacy_kfold` it will be ~96%.
4. §12 must print `logits after upsample: (1, 6, 512, 512)` — or `(1, 7, ...)` with
   `IGNORE_UNLABELLED = False`. A 128 in the spatial dims invalidates everything below it.
5. One fold per session, `FOLD = 0..4`. Budget 4-6 h per fold, two sessions each; resume is
   automatic from `last.pt`.
6. After all folds, run §21.

### Where this is not a literal transcription
Three are unavoidable, three are deliberate scoring choices with a named switch back.

| Deviation | Why | Effect on the number |
|---|---|---|
| `INTER_AREA` instead of bilinear for the 2345 -> 512 downsample | reuses the shared cache; bilinear aliases thin roads at 4.6x | small, in this notebook's **favour** |
| seeded RNG | the legacy script seeded nothing and was not reproducible | none in expectation |
| `use_deterministic_algorithms(False)` | 6x wall-clock otherwise; 300 epochs would not fit | none in expectation |
| `IGNORE_UNLABELLED = True` | annotation mistakes are not a class; matches the benchmark | should help the 6 real classes slightly |
| `VAL_USES_TRAIN_TF = False` | removes an augmentation lottery from checkpoint selection | small, reduces variance |
| `SELECTION_METRIC = "val_miou_core6"` | select on the metric you report, not a noisier proxy | small, reduces variance |

Everything else — split, weights, loss, optimizer, LR, schedule, clipping, EMA, epochs, patience,
batch, workers, augmentation, normalisation — is literal.

### Known residual weaknesses, inherited from the legacy recipe
These are not bugs; they are the recipe under test, and they are worth naming so the comparison in
the table below is read correctly. No clipping with uncapped inverse-frequency weights leaves the
occasional very large step (watch `grad_norm_true`). `drop_last=False` means the final train batch
holds 3 images and the final val batch 5, which is a noisy gradient and a noisy contribution to the
batch-averaged estimator. Class weights are counted over train+val together, a mild prior leak.
There is no EMA, typically worth +0.5-1.5 mIoU on a model this size. Changing any of them turns a
recipe comparison into something else, so change them only after run C has reported.

### The experiment
Three runs of this notebook, one field changed between them. Together they decompose the entire
0.63 -> 0.46 gap into named parts instead of leaving it as "something is wrong with the code".

| Run | Change | What it isolates |
|---|---|---|
| **A** | nothing (`legacy_kfold`, 5 folds) | reproduces 0.63 and confirms the transcription is faithful |
| **B** | `SPLIT_MODE = "random"`, `N_FOLDS = 3` | removes variant leakage, keeps random geography |
| **C** | `SPLIT_MODE = "block"`, `N_FOLDS = 3` | the reported protocol, legacy recipe |

Every run logs all four mIoU definitions, so within each one you also get the estimator effect for
free. Reading the result:

- **A's `legacy_miou_batch_7class` minus A's `mIoU_clean_core6`** = the metric and the 7th class.
- **A's `mIoU_clean_core6` minus B's** = variant leakage.
- **B minus C** = spatial geography.
- **C versus the 0.3962 already in your table** = the only genuine recipe comparison in the set.
  This is the number that decides whether flat 3e-5 for 300 epochs with plain weighted CE actually
  beats the current recipe, or whether the current recipe was never the problem.

Run C is the one that matters for the paper. If C comes out meaningfully above 0.3962, the present
recipe has a real deficiency and the most likely culprits, in order, are the frozen augmentation
from `persistent_workers=True`, the short cosine horizon relative to a run that was still
improving at epoch 300, and the 10x-lower encoder LR. If C lands near 0.3962, then the recipe was
fine all along and the 0.63 was never a segmentation result — it was a measurement of how much of
the validation set the model had already seen.

### One thing not to do
Do not put the legacy number in the paper, in either column, even with a footnote. A reviewer who
opens the training script will find `KFold` over 1224 files in about ninety seconds, and at that
point the question stops being about SegFormer-B0. The honest and genuinely more interesting
version of this story is the one you already have the data for: a benchmark where the split is
grouped and blocked, with a stated leakage gap. Adding "and here is what the same model scores
when the split is done naively" as a one-line methodological aside is a strength, not a weakness —
it is the kind of thing reviewers cite approvingly. It just has to be framed as a caution rather
than as a result.